# 💎 JSON Refiner Advanced Edition
### Professional Data Normalization & Transformation Suite

---

Run each cell **in order** (top to bottom) to launch the app.

| Cell | Purpose |
|------|---------|
| **Cell 1** | Install dependencies |
| **Cell 2** | Create `engine.py` (core logic) |
| **Cell 3** | Launch the App (Gradio UI) |

> ⚡ A **public shareable link** will be generated automatically.

In [ ]:
# ============================================================
# CELL 1 — Install Required Packages
# ============================================================
!pip install -q gradio>=4.0 json5>=0.9.6 jsonschema>=4.17.0
print('✅ Dependencies installed!')

In [ ]:
# ============================================================
# CELL 2 — Create engine.py
# ============================================================
engine_code = r'''import re
import json
import json5
import logging
from enum import Enum
from typing import Any, Dict, List, Optional, Union, Tuple
from jsonschema import validate, ValidationError, Draft7Validator

class CaseStyle(Enum):
    SNAKE_CASE = "snake_case"
    CAMEL_CASE = "camelCase"
    KEBAB_CASE = "kebab-case"
    PASCAL_CASE = "PascalCase"

class JSONRefinerCore:
    def __init__(self):
        self.stats = {
            "processed": 0,
            "errors": 0,
            "transformations": 0
        }
        self.logger = logging.getLogger("JSONRefiner")
        logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

    def log_error(self, message: str):
        self.logger.error(message)
        self.stats["errors"] += 1

    def infer_type(self, value: str) -> Any:
        """Infers the correct data type from a string value."""
        if not isinstance(value, str):
            return value
            
        val_lower = value.strip().lower()
        
        # Boolean check
        if val_lower in ['true', 'yes', 'on']:
            return True
        if val_lower in ['false', 'no', 'off']:
            return False
            
        # Null check
        if val_lower in ['null', 'none', 'nil', 'n/a']:
            return None
        if val_lower == '':
            return "" # Keep empty strings as empty strings instead of None
            
        # Number check
        try:
            if '.' in value:
                # Check if it's actually a number and not just a dot
                float_val = float(value)
                return float_val
            return int(value)
        except ValueError:
            pass
            
        # JSON string check (arrays or objects)
        if (value.strip().startswith('[') and value.strip().endswith(']')) or \
           (value.strip().startswith('{') and value.strip().endswith('}')):
            try:
                return json.loads(value)
            except:
                try:
                    return json5.loads(value)
                except:
                    pass
                    
        return value

    def to_snake_case(self, text: str) -> str:
        if not text: return text
        s1 = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', text)
        return re.sub('([a-z0-9])([A-Z])', r'\1_\2', s1).lower().replace("-", "_").replace(" ", "_")

    def to_camel_case(self, text: str) -> str:
        s = self.to_snake_case(text)
        parts = [p for p in s.split('_') if p]
        if not parts: return s
        return parts[0] + ''.join(i.title() for i in parts[1:])

    def to_pascal_case(self, text: str) -> str:
        s = self.to_snake_case(text)
        return ''.join(i.title() for i in s.split('_') if i)

    def to_kebab_case(self, text: str) -> str:
        return self.to_snake_case(text).replace("_", "-")

    def normalize_key(self, key: str, style: CaseStyle) -> str:
        if not isinstance(key, str): return key
        if style == CaseStyle.SNAKE_CASE:
            return self.to_snake_case(key)
        elif style == CaseStyle.CAMEL_CASE:
            return self.to_camel_case(key)
        elif style == CaseStyle.PASCAL_CASE:
            return self.to_pascal_case(key)
        elif style == CaseStyle.KEBAB_CASE:
            return self.to_kebab_case(key)
        return key

    def process_data(self, data: Any, case_style: Optional[CaseStyle] = None, 
                     infer_types: bool = True, remove_nulls: bool = False) -> Any:
        """Recursively processes JSON data with case normalization and type inference."""
        if isinstance(data, dict):
            new_dict = {}
            for k, v in data.items():
                new_key = self.normalize_key(k, case_style) if case_style else k
                processed_val = self.process_data(v, case_style, infer_types, remove_nulls)
                
                if remove_nulls and (processed_val is None or processed_val == ""):
                    continue
                new_dict[new_key] = processed_val
            return new_dict
        elif isinstance(data, list):
            res = [self.process_data(i, case_style, infer_types, remove_nulls) for i in data]
            if remove_nulls:
                res = [i for i in res if i is not None and i != ""]
            return res
        else:
            return self.infer_type(data) if infer_types else data

    def validate_json_schema(self, data: Any, schema: Dict[str, Any]) -> Tuple[bool, Optional[str]]:
        """Validates JSON data against a master schema."""
        try:
            Draft7Validator.check_schema(schema)
            validate(instance=data, schema=schema)
            return True, None
        except ValidationError as e:
            return False, e.message
        except Exception as e:
            return False, str(e)

    def check_required_fields(self, data: Dict[str, Any], required_fields: List[str]) -> List[str]:
        """Checks for missing required fields in data."""
        missing = []
        if not isinstance(data, dict): return []
        for field in required_fields:
            if field not in data:
                missing.append(field)
        return missing

    def flatten_json(self, data: Any, parent_key: str = '', sep: str = '.') -> Dict[str, Any]:
        """Flattens a nested JSON structure into dot-notation."""
        items = []
        if isinstance(data, dict):
            for k, v in data.items():
                new_key = f"{parent_key}{sep}{k}" if parent_key else k
                if isinstance(v, (dict, list)):
                    items.extend(self.flatten_json(v, new_key, sep=sep).items())
                else:
                    items.append((new_key, v))
        elif isinstance(data, list):
            for i, v in enumerate(data):
                new_key = f"{parent_key}{sep}{i}" if parent_key else str(i)
                if isinstance(v, (dict, list)):
                    items.extend(self.flatten_json(v, new_key, sep=sep).items())
                else:
                    items.append((new_key, v))
        else:
            return {parent_key: data}
        return dict(items)

    def unflatten_json(self, data: Dict[str, Any], sep: str = '.') -> Dict[str, Any]:
        """Unflattens dot-notation keys back into a nested structure."""
        if not isinstance(data, dict): return data
        result = {}
        for key, value in data.items():
            parts = key.split(sep)
            d = result
            for part in parts[:-1]:
                if part not in d:
                    d[part] = {}
                d = d[part]
            d[parts[-1]] = value
        return result

    def merge_json_objects(self, obj1: Any, obj2: Any) -> Any:
        """Deep merges two JSON objects or lists."""
        if isinstance(obj1, dict) and isinstance(obj2, dict):
            result = obj1.copy()
            for key, value in obj2.items():
                if key in result:
                    result[key] = self.merge_json_objects(result[key], value)
                else:
                    result[key] = value
            return result
        elif isinstance(obj1, list) and isinstance(obj2, list):
            return obj1 + obj2
        else:
            return obj2

    def remove_null_values(self, data: Any) -> Any:
        """Helper to specifically remove null values without other processing."""
        return self.process_data(data, infer_types=False, remove_nulls=True)

    def parse_key_value_text(self, text: str, case_style: Optional[CaseStyle] = None, infer_types: bool = True) -> Tuple[Dict[str, Any], List[str]]:
        """Parses key-value pair text into a dictionary."""
        result = {}
        errors = []
        lines = text.split('\n')
        
        for line_num, line in enumerate(lines, 1):
            line = line.strip()
            if not line or line.startswith(('#', '//')):
                continue
                
            if ':' not in line:
                errors.append(f"Line {line_num}: No colon separator found - '{line}'")
                continue
                
            try:
                key, value = line.split(':', 1)
                key = key.strip()
                value = value.strip()
                
                if case_style:
                    key = self.normalize_key(key, case_style)
                    
                result[key] = self.infer_type(value) if infer_types else value
            except Exception as e:
                errors.append(f"Line {line_num}: Error parsing - {str(e)}")
                
        return result, errors

    def check_required_fields_detailed(self, data: Dict[str, Any], required_fields: List[str]) -> Tuple[bool, List[str]]:
        """Checks for mandatory fields and returns status and missing list."""
        if not isinstance(data, dict):
            return False, ["Input is not an object"]
        missing = [f for f in required_fields if f not in data]
        if missing:
            return False, missing
        return True, []

'''

with open('engine.py', 'w', encoding='utf-8') as f:
    f.write(engine_code)

print('✅ engine.py written successfully!')

In [ ]:
# ============================================================
# CELL 3 — Launch the App
# ============================================================
import gradio as gr
import json
import json5
import datetime
import os
from engine import JSONRefinerCore, CaseStyle

# Initialize Core Engine
core = JSONRefinerCore()
history_log = []

def get_timestamp():
    return datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def log_action(action_type, details, status="Success"):
    history_log.append({
        "timestamp": get_timestamp(),
        "action": action_type,
        "details": details,
        "status": status
    })

# --- TAB 1: Core Refining Logic ---
def core_refine(text, style_name, infer_types, remove_nulls, flatten):
    try:
        # Try JSON first
        try:
            # Use json5 to be more flexible (handles comments, trailing commas)
            data = json5.loads(text)
            errors = []
        except:
            # Fallback to Key-Value parsing
            data, errors = core.parse_key_value_text(text, infer_types=infer_types)
            
        if not data and errors:
            log_action("Core Refining", "Failed parsing", "Error")
            return f"❌ Errors found:\n" + "\n".join(errors), "Error"
        
        style = getattr(CaseStyle, style_name.upper().replace(" ", "_"), None) if style_name != "None" else None
        processed = core.process_data(data, case_style=style, infer_types=infer_types, remove_nulls=remove_nulls)
        
        if flatten:
            processed = core.flatten_json(processed)
            
        log_action("Core Refining", f"Processed input", "Success")
        output = json.dumps(processed, indent=4)
        return output, f"✅ Processed successfully. " + (f"⚠️ {len(errors)} errors in KV parsing." if errors else "")
    except Exception as e:
        log_action("Core Refining", f"Error: {str(e)}", "Error")
        return f"❌ Unexpected Error: {str(e)}", "Error"

# --- TAB 2: Validation Logic ---
def validate_json(json_text, schema_text, required_fields_str):
    try:
        data = json5.loads(json_text)
        status_msgs = []
        
        if schema_text.strip():
            schema = json.loads(schema_text)
            is_valid, error = core.validate_json_schema(data, schema)
            status_msgs.append("✅ Schema Valid" if is_valid else f"❌ Schema Invalid: {error}")
            
        if required_fields_str.strip():
            req_fields = [f.strip() for f in required_fields_str.split(",") if f.strip()]
            is_ok, missing = core.check_required_fields_detailed(data, req_fields)
            status_msgs.append("✅ All required fields present" if is_ok else f"❌ Missing: {', '.join(missing)}")
            
        log_action("Validation", "Schema & Fields check", "Checked")
        return "\n\n".join(status_msgs) if status_msgs else "No validation criteria provided."
    except Exception as e:
        log_action("Validation", f"Error: {str(e)}", "Error")
        return f"❌ Error: {str(e)}"

# --- TAB 3: Case Conversion Logic ---
def convert_case(json_text, target_style):
    try:
        data = json5.loads(json_text)
        style = getattr(CaseStyle, target_style.upper().replace(" ", "_"), None)
        processed = core.process_data(data, case_style=style, infer_types=False)
        log_action("Case Conversion", f"To {target_style}", "Success")
        return json.dumps(processed, indent=4)
    except Exception as e:
        log_action("Case Conversion", f"Error: {str(e)}", "Error")
        return f"❌ Error: {str(e)}"

# --- TAB 4: Merge Logic ---
def merge_json(json1, json2, json3):
    try:
        objs = []
        for j in [json1, json2, json3]:
            if j.strip():
                objs.append(json5.loads(j))
        
        if not objs: return "No input provided."
        
        result = objs[0]
        for other in objs[1:]:
            result = core.merge_json_objects(result, other)
            
        log_action("Merging", f"Merged {len(objs)} objects", "Success")
        return json.dumps(result, indent=4)
    except Exception as e:
        log_action("Merging", f"Error: {str(e)}", "Error")
        return f"❌ Error: {str(e)}"

# --- TAB 5: Flatten Logic ---
def toggle_nesting(json_text, mode):
    try:
        data = json5.loads(json_text)
        if mode == "Flatten":
            res = core.flatten_json(data)
        else:
            res = core.unflatten_json(data)
        log_action(f"{mode}", "Structural change", "Success")
        return json.dumps(res, indent=4)
    except Exception as e:
        log_action(f"{mode}", f"Error: {str(e)}", "Error")
        return f"❌ Error: {str(e)}"

# --- TAB 6: History Logic ---
def get_history_text():
    if not history_log: return "No history yet."
    output = ""
    for item in reversed(history_log):
        output += f"[+] Time: {item['timestamp']}\n    Action: {item['action']}\n    Details: {item['details']}\n    Status: {item['status']}\n"
        output += "-"*40 + "\n"
    return output

def download_history():
    content = get_history_text()
    path = "transformation_history.txt"
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)
    return path

def clear_history():
    global history_log
    history_log = []
    return "History cleared."

# CSS
custom_css = """
.gradio-container { background: #0f172a; color: white; font-family: 'Inter', sans-serif; }
.bento-card { background: rgba(30,41,59,0.7); backdrop-filter: blur(12px); border: 1px solid rgba(255,255,255,0.1); border-radius: 18px; padding: 25px; margin: 15px 0; }
.primary-btn { background: linear-gradient(135deg, #6366f1, #a855f7) !important; color: white !important; border-radius: 10px !important; font-weight: bold !important; transition: all 0.3s ease !important; }
.primary-btn:hover { transform: translateY(-2px); box-shadow: 0 8px 20px -5px #6366f1; }
.secondary-btn { background: rgba(255,255,255,0.1) !important; color: white !important; border-radius: 10px !important; }
textarea, input { background: rgba(255,255,255,0.04) !important; color: white !important; border: 1px solid rgba(255,255,255,0.1) !important; border-radius: 8px !important; }
.tabs { margin-top: 25px; border-radius: 15px; overflow: hidden; }
"""

with gr.Blocks(theme=gr.themes.Soft(primary_hue="indigo"), css=custom_css) as demo:
    gr.Markdown("# 💎 JSON Refiner Advanced Edition\n### Professional Data Normalization & Transformation Suite")
    
    with gr.Tabs(elem_classes="tabs"):
        # 1. Core Refining
        with gr.Tab("⚒️ Core Refining"):
            with gr.Row():
                with gr.Column(scale=3):
                    input_kv = gr.Textbox(label="Key-Value Format / Source Data", lines=12, 
                                        placeholder="name: John Doe\nage: 28\nemail: john@example.com",
                                        value="name: John Doe\nage: 28\nactive: true\nemail: john@example.com")
                with gr.Column(scale=2):
                    with gr.Group(elem_classes="bento-card"):
                        gr.Markdown("### Configuration")
                        case_sel = gr.Radio(["None", "Snake Case", "Camel Case", "Pascal Case", "Kebab Case"], 
                                          label="Key Normalization", value="Snake Case")
                        inf_type = gr.Checkbox(label="Auto-Infer Types (bool, numbers, null)", value=True)
                        rem_null = gr.Checkbox(label="Remove Null/Empty Values", value=False)
                        flat_chk = gr.Checkbox(label="Flatten Output (Dot notation)", value=False)
                    refine_btn = gr.Button("🚀 REFINE JSON", variant="primary", elem_classes="primary-btn")
            
            with gr.Row():
                output_core = gr.Code(label="Refined Result", language="json")
                status_core = gr.Markdown("Status: Ready")
            
            refine_btn.click(core_refine, [input_kv, case_sel, inf_type, rem_null, flat_chk], [output_core, status_core])

        # 2. Validation
        with gr.Tab("✅ Validation"):
            with gr.Row():
                with gr.Column():
                    val_input = gr.Code(label="JSON to Validate", language="json", value='{"name": "John Doe", "age": 28, "email": "john@example.com"}')
                    val_schema = gr.Textbox(label="JSON Schema (Optional)", lines=6, placeholder='{"type": "object", "properties": {"name": {"type": "string"}}}')
                    val_req = gr.Textbox(label="Required Fields (comma separated)", placeholder="name, age, email", value="name, age")
                    val_btn = gr.Button("🔍 Validate Structure", variant="primary", elem_classes="primary-btn")
                with gr.Column():
                    val_results = gr.Markdown("### Validation Results\nResults will be shown here after checking...")
            val_btn.click(validate_json, [val_input, val_schema, val_req], val_results)

        # 3. Case Conversion
        with gr.Tab("🔡 Case Conversion"):
            with gr.Row():
                with gr.Column():
                    case_input = gr.Code(label="Source JSON", language="json", value='{"first_name": "John", "last_name": "Doe", "user_age": 28}')
                    case_target = gr.Radio(["Snake Case", "Camel Case", "Pascal Case", "Kebab Case"], label="Target Convention", value="Camel Case")
                    case_btn = gr.Button("🔄 Convert Style", variant="primary", elem_classes="primary-btn")
                with gr.Column():
                    case_output = gr.Code(label="Converted Output", language="json")
            case_btn.click(convert_case, [case_input, case_target], case_output)

        # 4. Merge JSON
        with gr.Tab("🔗 Merge JSON"):
            gr.Markdown("### Combine Multiple Data Sources (Deep Merge)")
            with gr.Row():
                m1 = gr.Code(label="Primary Object", language="json", value='{"user": {"name": "John"}}')
                m2 = gr.Code(label="Secondary Object", language="json", value='{"user": {"age": 28}}')
                m3 = gr.Code(label="Tertiary Object (Optional)", language="json", value='{"active": true}')
            merge_btn = gr.Button("🤝 Deep Merge Objects", variant="primary", elem_classes="primary-btn")
            merge_out = gr.Code(label="Merged Result", language="json")
            merge_btn.click(merge_json, [m1, m2, m3], merge_out)

        # 5. Flatten/Unflatten
        with gr.Tab("📦 Flatten/Unflatten"):
            with gr.Row():
                with gr.Column():
                    flat_input = gr.Code(label="Source JSON", language="json", value='{"user": {"profile": {"name": "John", "details": {"active": true}}}}')
                    flat_mode = gr.Radio(["Flatten", "Unflatten"], label="Transformation Mode", value="Flatten")
                    flat_btn = gr.Button("⚡ Transform Nesting", variant="primary", elem_classes="primary-btn")
                with gr.Column():
                    flat_output = gr.Code(label="Transformed Output", language="json")
            flat_btn.click(toggle_nesting, [flat_input, flat_mode], flat_output)

        # 6. History
        with gr.Tab("📜 History"):
            with gr.Row():
                with gr.Column():
                    history_display = gr.Textbox(label="Transformation Log", lines=20, value=get_history_text, interactive=False)
                with gr.Column():
                    gr.Markdown("### Management")
                    refresh_hist = gr.Button("🔄 Refresh Log", elem_classes="secondary-btn")
                    download_btn = gr.Button("📥 Download History (.txt)", variant="primary", elem_classes="primary-btn")
                    clear_hist = gr.Button("🗑️ Reset History", variant="stop")
                    file_out = gr.File(label="Generated File")
            
            refresh_hist.click(get_history_text, None, history_display)
            download_btn.click(download_history, None, file_out)
            clear_hist.click(clear_history, None, history_display)

        # 7. Guide
        with gr.Tab("📖 Guide"):
            gr.Markdown("""
            # 💎 JSON Refiner - Complete Guide
            
            ### ■ Input Format
            The system accepts two types of input:
            1. **Standard JSON**: Valid JSON objects or arrays.
            2. **Key:Value Text**: Simple lines formatted as `key: value`.
            
            ### ■ Type Inference
            Automatic detection of:
            - `true`, `false`, `yes`, `no` -> Boolean
            - `null`, `none`, `nil` -> Null
            - `123`, `45.6` -> Numbers (Integer/Float)
            - `[...]`, `{...}` -> Nested Objects/Arrays
            - Everything else -> String
            
            ### ■ Case Styles Supported
            - **Snake Case**: `user_first_name`
            - **Camel Case**: `userFirstName`
            - **Pascal Case**: `UserFirstName`
            - **Kebab Case**: `user-first-name`
            
            ### ■ Key Features
            - **Flattening**: Useful for converting nested JSON into a flat structure for Excel/CSV.
            - **Merging**: Combine fragmented data from multiple API responses.
            - **Validation**: Ensure production readiness using JSON Schema.
            """)

if __name__ == "__main__":
    demo.launch(share=True)

# Ensure public sharing is enabled
if 'demo' in locals():
    demo.launch(share=True)